# OpenRLHF PPO

## Scheme

```python
# one episode: traverse the prompt data once
for episode in range(args.num_episodes):

    # len(rand_prompts) = micro_rollout_batch_size
    for rand_prompts in self.prompts_dataloader:
        # make experience from Actor
        experience = self.experience_maker.make_experience(rand_prompts, **self.generate_kwargs)
        self.replay_buffer.append(experience)

        # update_timesteps = args.rollout_batch_size // (self.strategy.world_size * self.micro_rollout_batch_size)
        # if rollout_batch_size = 512, world_size = 8, micro_rollout_batch_size = 16
        # then update_timesteps = 512 // (8 * 16) = 4
        if steps % update_timesteps == 0:

            # one step -> one micro roll out
            # one global step -> one micro train
            global_steps = steps // update_timesteps
            
            self.replay_buffer.normalize("advantages", self.strategy)
            status = self.ppo_train(global_steps)
            self.replay_buffer.clear()

        steps = steps + 1
```

## `ppo_train`

```python
# replay_buffer.sample_batch_size = micro_train_batch_size, i.e. batch size per GPU
dataloader = DataLoader(
    self.replay_buffer, 
    batch_size=self.replay_buffer.sample_batch_size, 
    shuffle=True,
    ...
)

# use data multiple times
for epoch in range(self.max_epochs):

    # len(experience) = micro_train_batch_size
    for experience in dataloader:
        
        status = {}
        status = self.training_step_actor(experience)
        status.update(self.training_step_critic(experience))
```

## `training_step_actor`

```python
log_probs, output = self.actor(experience.sequences, ...)

# PPO loss
ratio = (log_probs - experience.log_probs).exp()
surr1 = ratio * experience.advantages
surr2 = ratio.clamp(1 - self.clip_eps, 1 + self.clip_eps) * advantages
loss = -torch.min(surr1, surr2)

self.strategy.backward(loss, self.actor, self.actor_optim)
self.strategy.optimizer_step(self.actor_optim, self.actor, self.actor_scheduler, name="actor")
```

## `training_step_critic`

```python
values, output = self.critic(experience.sequences, ...)

loss = (values - experience.returns) ** 2

self.strategy.backward(loss, self.critic, self.critic_optim)
self.strategy.optimizer_step(self.critic_optim, self.critic, self.critic_scheduler, name="critic")
```